<a href="https://colab.research.google.com/github/cylaadhan/FuzzyLogic/blob/main/word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install gensim openpyxl

  Using cached gensim-4.4.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
Using cached gensim-4.4.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (27.9 MB)


In [4]:
import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
matrik = pd.read_excel("matrik.xlsx")
standar = pd.read_excel("standar.xlsx")

In [6]:
print("Kolom matrik:")
print(matrik.columns)

print("\nKolom standar:")
print(standar.columns)

Kolom matrik:
Index(['KRITERIA', 'ELEMEN', 'DESKRIPTOR', 'FULL'], dtype='object')

Kolom standar:
Index(['STANDAR', 'PERNYATAAN ISI STANDAR', 'INDIKATOR', 'FULL'], dtype='object')


In [7]:
kolom_matrik = "FULL"
kolom_standar = "FULL"

In [8]:
matrik = matrik.dropna(subset=[kolom_matrik]).reset_index(drop=True)
standar = standar.dropna(subset=[kolom_standar]).reset_index(drop=True)

print("Jumlah data matrik:", len(matrik))
print("Jumlah data standar:", len(standar))

Jumlah data matrik: 75
Jumlah data standar: 509


In [9]:
def bersihkan_teks(teks):
    teks = str(teks).lower()
    teks = teks.replace("&nbsp;", " ")
    teks = re.sub(r"\[.*?\]", " ", teks)
    teks = re.sub(r"[^a-zA-Z0-9\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks)
    teks = teks.strip()
    return teks

In [10]:
matrik["teks_bersih"] = matrik[kolom_matrik].apply(bersihkan_teks)
standar["teks_bersih"] = standar[kolom_standar].apply(bersihkan_teks)

In [11]:
matrik[["teks_bersih"]].head()

,teks_bersih
0,budaya mutu kebijakan standar dan indikator te...
1,budaya mutu kebijakan standar dan indikator te...
2,budaya mutu efektifitas pelaksanaan kegiatan t...
3,budaya mutu efektifitas pelaksanaan standar da...
4,budaya mutu efektifitas keberkalaan pelaksanaa...


In [12]:
standar[["teks_bersih"]].head()

,teks_bersih
0,standar kompetensi lulusan sk rektor tentang k...
1,standar kompetensi lulusan dokumen tentang kom...
2,standar kompetensi lulusan dokumen tentang kua...
3,standar kompetensi lulusan dokumen tentang kom...
4,standar kompetensi lulusan dokumen rencana pem...


In [13]:
matrik["tokens"] = matrik["teks_bersih"].apply(lambda x: x.split())
standar["tokens"] = standar["teks_bersih"].apply(lambda x: x.split())

In [14]:
matrik[["tokens"]].head()

,tokens
0,"[budaya, mutu, kebijakan, standar, dan, indika..."
1,"[budaya, mutu, kebijakan, standar, dan, indika..."
2,"[budaya, mutu, efektifitas, pelaksanaan, kegia..."
3,"[budaya, mutu, efektifitas, pelaksanaan, stand..."
4,"[budaya, mutu, efektifitas, keberkalaan, pelak..."


In [15]:
standar[["tokens"]].head()

,tokens
0,"[standar, kompetensi, lulusan, sk, rektor, ten..."
1,"[standar, kompetensi, lulusan, dokumen, tentan..."
2,"[standar, kompetensi, lulusan, dokumen, tentan..."
3,"[standar, kompetensi, lulusan, dokumen, tentan..."
4,"[standar, kompetensi, lulusan, dokumen, rencan..."


In [16]:
semua_tokens = pd.concat([
    matrik["tokens"],
    standar["tokens"]
], axis=0).tolist()

In [17]:
semua_tokens[:3]

[['budaya',
  'mutu',
  'kebijakan',
  'standar',
  'dan',
  'indikator',
  'terkait',
  'sistem',
  'tata',
  'kelola',
  'internal',
  'upps',
  'dan',
  'atau',
  'pt',
  'berikut',
  'sop',
  'yang',
  'mencakup',
  'administrasi',
  'akademik',
  'keuangan',
  'sdm',
  'dan',
  'aspek',
  'lain',
  'dalam',
  'siklus',
  'ppepp',
  'di',
  'tingkat',
  'upps',
  'dan',
  'atau',
  'pt',
  'ketersediaan',
  'kebijakan',
  'standar',
  'dan',
  'indikator',
  'terkait',
  'sistem',
  'tata',
  'kelola',
  'internal',
  'upps',
  'dan',
  'atau',
  'pt',
  'berikut',
  'sop',
  'yang',
  'mencakup',
  'administrasi',
  'akademik',
  'keuangan',
  'sdm',
  'dan',
  'aspek',
  'lain',
  'dalam',
  'siklus',
  'ppepp',
  'di',
  'tingkat',
  'upps',
  'dan',
  'atau',
  'pt'],
 ['budaya',
  'mutu',
  'kebijakan',
  'standar',
  'dan',
  'indikator',
  'terkait',
  'fungsi',
  'spmi',
  'dengan',
  'sdm',
  'yang',
  'kompeten',
  'sebagai',
  'pelaksana',
  'di',
  'tingkat',
  'upps',


In [18]:
model_w2v = Word2Vec(
    sentences=semua_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100
)

In [19]:
list(model_w2v.wv.index_to_key)[:20]

['dan',
 'standar',
 'yang',
 'penelitian',
 'masyarakat',
 'dokumen',
 'setiap',
 'rektor',
 'pengabdian',
 'tentang',
 'harus',
 'sarana',
 'dosen',
 'hasil',
 'prasarana',
 'tahun',
 'pembelajaran',
 'pkm',
 'tata',
 'kepada']

In [20]:
model_w2v.wv["standar"]

array([ 6.9138547e-04,  2.1260807e-01,  3.4788600e-01, -2.5232987e-02,
       -1.3269709e-01, -7.3862690e-01, -1.5160234e-01,  6.6644216e-01,
       -8.5910058e-01, -5.1946938e-02, -4.0140599e-02,  2.7418324e-01,
       -3.5244030e-01,  7.5272709e-01, -1.7768575e-01, -2.0350236e-01,
        3.6993566e-01,  1.9323301e-01,  2.2528735e-03, -2.0389536e-01,
       -9.8905936e-02, -3.9540540e-02,  2.9484600e-01, -5.3857666e-01,
        1.2395934e-01,  5.0131552e-02,  2.7864254e-01, -2.2247199e-02,
       -7.9374707e-01,  4.8666289e-01,  3.0861530e-01, -3.0708608e-01,
        4.2972520e-01, -4.8744866e-01, -4.8879508e-02,  5.1067555e-01,
        8.5533567e-02, -2.8446233e-01, -5.2499777e-01,  5.8553807e-02,
        5.3442156e-01, -3.1312132e-01, -3.3836263e-01, -2.4541244e-01,
        2.7103516e-01,  6.2983584e-01, -8.4511948e-01,  5.3085208e-01,
       -1.9356342e-01,  1.1700635e-01,  2.0342758e-01, -2.6870078e-01,
        1.0141525e-01, -4.3863177e-01, -1.8183416e-01,  3.7889519e-01,
      

In [21]:
def kalimat_ke_vektor(tokens, model):
    vektor_kata = []

    for kata in tokens:
        if kata in model.wv:
            vektor_kata.append(model.wv[kata])

    if len(vektor_kata) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vektor_kata, axis=0)

In [22]:
contoh_vektor = kalimat_ke_vektor(matrik.loc[0, "tokens"], model_w2v)

print(contoh_vektor)
print("Ukuran vector:", contoh_vektor.shape)

[ 0.22324304 -0.1180537   0.15694004 -0.2725996   0.40276018  0.1113462
 -0.12515879  0.44791946 -0.62214255  0.22090735 -0.16935739 -0.38506037
 -0.16505624  0.6570557   0.09516092 -0.13118103  0.00281903 -0.2930322
 -0.05027487 -0.13133402  0.23487175 -0.20863132 -0.00288319 -0.06329068
  0.10461223  0.0368007   0.28649026  0.0837584  -0.11595041  0.07263555
  0.17475985 -0.35520318 -0.18026891  0.02349174  0.01905734  0.38141325
 -0.40322685 -0.42473474 -0.14556436 -0.29607075  0.03860208 -0.38952884
 -0.42033684 -0.20961685 -0.23466417  0.10062681  0.16365324  0.33807036
  0.21915682 -0.139992   -0.12135141  0.13015927 -0.27678776 -0.543589
 -0.17862985  0.20506167 -0.21968256 -0.12305357 -0.20286836 -0.12096418
 -0.28514344  0.02585455  0.18242927 -0.09269919 -0.47237867  0.01751679
 -0.28687343  0.11607144 -0.22165924  0.29736322  0.48006257 -0.27453312
 -0.00223094 -0.5405568   0.2549899  -0.0148524  -0.436924    0.32877567
 -0.22283736  0.37039706 -0.08877724  0.22468783  0.085

In [23]:
vektor_matrik = np.array([
    kalimat_ke_vektor(tokens, model_w2v)
    for tokens in matrik["tokens"]
])

vektor_standar = np.array([
    kalimat_ke_vektor(tokens, model_w2v)
    for tokens in standar["tokens"]
])

In [24]:
print("Bentuk vektor matrik:", vektor_matrik.shape)
print("Bentuk vektor standar:", vektor_standar.shape)

Bentuk vektor matrik: (75, 100)
Bentuk vektor standar: (509, 100)


In [25]:
similarity_matrix = cosine_similarity(vektor_matrik, vektor_standar)

In [26]:
similarity_matrix = cosine_similarity(vektor_matrik, vektor_standar)

In [27]:
hasil = []

for i in range(len(matrik)):
    nilai_similarity = similarity_matrix[i]

    index_terbaik = nilai_similarity.argmax()
    skor_terbaik = nilai_similarity[index_terbaik]

    hasil.append({
        "No_Matrik": i + 1,
        "Kriteria_Matrik": matrik.loc[i, "KRITERIA"] if "KRITERIA" in matrik.columns else "",
        "Elemen_Matrik": matrik.loc[i, "ELEMEN"] if "ELEMEN" in matrik.columns else "",
        "Deskriptor_Matrik": matrik.loc[i, "DESKRIPTOR"] if "DESKRIPTOR" in matrik.columns else "",
        "Teks_Matrik": matrik.loc[i, kolom_matrik],

        "No_Standar_Terpilih": index_terbaik + 1,
        "Standar_Terpilih": standar.loc[index_terbaik, "Standar"] if "Standar" in standar.columns else "",
        "Pernyataan_Standar": standar.loc[index_terbaik, "Pernyataan Isi Standar"] if "Pernyataan Isi Standar" in standar.columns else "",
        "Indikator_Standar": standar.loc[index_terbaik, "Indikator"] if "Indikator" in standar.columns else "",
        "Teks_Standar_Terpilih": standar.loc[index_terbaik, kolom_standar],

        "Similarity_Word2Vec_Cosine": skor_terbaik
    })

hasil_df = pd.DataFrame(hasil)

In [28]:
hasil_df.head(10)

,No_Matrik,Kriteria_Matrik,Elemen_Matrik,Deskriptor_Matrik,Teks_Matrik,No_Standar_Terpilih,Standar_Terpilih,Pernyataan_Standar,Indikator_Standar,Teks_Standar_Terpilih,Similarity_Word2Vec_Cosine
0,1,Budaya Mutu,"Kebijakan, standar, dan\nindikator terkait sis...","Ketersediaan\nkebijakan, standar,\ndan indikat...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.724339
1,2,Budaya Mutu,"Kebijakan, standar dan\nindikator terkait fung...","Ketersediaan\nkebijakan,standar dan\nindikator...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.729054
2,3,Budaya Mutu,Efektifitas pelaksanaan kegiatan terkait siste...,Efektifitas pelaksanaan \nkegiatan terkait \ns...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.764388
3,4,Budaya Mutu,Efektifitas pelaksanaan standar dan indikator ...,Efektifitas pelaksanaan kegiatan terkait stand...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.724490
4,5,Budaya Mutu,Efektifitas keberkalaan pelaksanaan evaluasi k...,Efektifitas dan keberkalaan pelaksanaan evalua...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas ke...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.740525
5,6,Budaya Mutu,Efektifitas pelaksanaan evaluasi ketercapaian ...,Efektifitas pelaksanaan evaluasi ketercapaian ...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.695694
6,7,Budaya Mutu,Efektifitas pelaksanaan tindak lanjut hasil ev...,Efektifitas pelaksanaan tindak lanjut hasil ev...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.758305
7,8,Budaya Mutu,Efektifitas pelaksanaan tindak lanjut hasil ev...,Efektifitas pelaksanaan tindak lanjut hasil ev...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.693605
8,9,Budaya Mutu,Efektifitas peningkatan/ optimalisasi standar ...,Efektifitas Peningkatan/optimalisasi standar d...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.741837
9,10,Budaya Mutu,Efektifitas peningkatan/ optimalisasi standar ...,Efektifitas Peningkatan/optimalisasi standar d...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.713489


In [29]:
def kategori_similarity(nilai):
    if nilai >= 0.80:
        return "Sangat Mirip"
    elif nilai >= 0.60:
        return "Mirip"
    elif nilai >= 0.40:
        return "Cukup Mirip"
    elif nilai >= 0.20:
        return "Kurang Mirip"
    else:
        return "Tidak Mirip"

hasil_df["Kategori"] = hasil_df["Similarity_Word2Vec_Cosine"].apply(kategori_similarity)

In [30]:
hasil_df.head(10)

,No_Matrik,Kriteria_Matrik,Elemen_Matrik,Deskriptor_Matrik,Teks_Matrik,No_Standar_Terpilih,Standar_Terpilih,Pernyataan_Standar,Indikator_Standar,Teks_Standar_Terpilih,Similarity_Word2Vec_Cosine,Kategori
0,1,Budaya Mutu,"Kebijakan, standar, dan\nindikator terkait sis...","Ketersediaan\nkebijakan, standar,\ndan indikat...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.724339,Mirip
1,2,Budaya Mutu,"Kebijakan, standar dan\nindikator terkait fung...","Ketersediaan\nkebijakan,standar dan\nindikator...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.729054,Mirip
2,3,Budaya Mutu,Efektifitas pelaksanaan kegiatan terkait siste...,Efektifitas pelaksanaan \nkegiatan terkait \ns...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.764388,Mirip
3,4,Budaya Mutu,Efektifitas pelaksanaan standar dan indikator ...,Efektifitas pelaksanaan kegiatan terkait stand...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.724490,Mirip
4,5,Budaya Mutu,Efektifitas keberkalaan pelaksanaan evaluasi k...,Efektifitas dan keberkalaan pelaksanaan evalua...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas ke...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.740525,Mirip
5,6,Budaya Mutu,Efektifitas pelaksanaan evaluasi ketercapaian ...,Efektifitas pelaksanaan evaluasi ketercapaian ...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.695694,Mirip
6,7,Budaya Mutu,Efektifitas pelaksanaan tindak lanjut hasil ev...,Efektifitas pelaksanaan tindak lanjut hasil ev...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.758305,Mirip
7,8,Budaya Mutu,Efektifitas pelaksanaan tindak lanjut hasil ev...,Efektifitas pelaksanaan tindak lanjut hasil ev...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.693605,Mirip
8,9,Budaya Mutu,Efektifitas peningkatan/ optimalisasi standar ...,Efektifitas Peningkatan/optimalisasi standar d...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.741837,Mirip
9,10,Budaya Mutu,Efektifitas peningkatan/ optimalisasi standar ...,Efektifitas Peningkatan/optimalisasi standar d...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.713489,Mirip


In [31]:
hasil_df.to_excel("hasil_word2vec_cosine_similarity.xlsx", index=False)

In [33]:
from google.colab import files
files.download("hasil_word2vec_cosine_similarity.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
hasil_top3 = []

for i in range(len(matrik)):
    nilai_similarity = similarity_matrix[i]

    top3_index = nilai_similarity.argsort()[-3:][::-1]

    for rank, idx in enumerate(top3_index, start=1):
        hasil_top3.append({
            "No_Matrik": i + 1,
            "Rank": rank,
            "Kriteria_Matrik": matrik.loc[i, "KRITERIA"] if "KRITERIA" in matrik.columns else "",
            "Elemen_Matrik": matrik.loc[i, "ELEMEN"] if "ELEMEN" in matrik.columns else "",
            "Deskriptor_Matrik": matrik.loc[i, "DESKRIPTOR"] if "DESKRIPTOR" in matrik.columns else "",
            "Teks_Matrik": matrik.loc[i, kolom_matrik],

            "No_Standar_Terpilih": idx + 1,
            "Standar_Terpilih": standar.loc[idx, "Standar"] if "Standar" in standar.columns else "",
            "Pernyataan_Standar": standar.loc[idx, "Pernyataan Isi Standar"] if "Pernyataan Isi Standar" in standar.columns else "",
            "Indikator_Standar": standar.loc[idx, "Indikator"] if "Indikator" in standar.columns else "",
            "Teks_Standar_Terpilih": standar.loc[idx, kolom_standar],

            "Similarity_Word2Vec_Cosine": nilai_similarity[idx]
        })

hasil_top3_df = pd.DataFrame(hasil_top3)

hasil_top3_df["Kategori"] = hasil_top3_df["Similarity_Word2Vec_Cosine"].apply(kategori_similarity)

hasil_top3_df.head(15)

,No_Matrik,Rank,Kriteria_Matrik,Elemen_Matrik,Deskriptor_Matrik,Teks_Matrik,No_Standar_Terpilih,Standar_Terpilih,Pernyataan_Standar,Indikator_Standar,Teks_Standar_Terpilih,Similarity_Word2Vec_Cosine,Kategori
0,1,1,Budaya Mutu,"Kebijakan, standar, dan\nindikator terkait sis...","Ketersediaan\nkebijakan, standar,\ndan indikat...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.724339,Mirip
1,1,2,Budaya Mutu,"Kebijakan, standar, dan\nindikator terkait sis...","Ketersediaan\nkebijakan, standar,\ndan indikat...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",43,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.718226,Mirip
2,1,3,Budaya Mutu,"Kebijakan, standar, dan\nindikator terkait sis...","Ketersediaan\nkebijakan, standar,\ndan indikat...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",304,,,,[STANDAR] Standar Tata Kelola dan Tata Pamong ...,0.698304,Mirip
3,2,1,Budaya Mutu,"Kebijakan, standar dan\nindikator terkait fung...","Ketersediaan\nkebijakan,standar dan\nindikator...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.729054,Mirip
4,2,2,Budaya Mutu,"Kebijakan, standar dan\nindikator terkait fung...","Ketersediaan\nkebijakan,standar dan\nindikator...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",205,,,,[STANDAR] Standar Pengelolaan Pengabdian Masya...,0.681374,Mirip
5,2,3,Budaya Mutu,"Kebijakan, standar dan\nindikator terkait fung...","Ketersediaan\nkebijakan,standar dan\nindikator...","[KRITERIA] Budaya Mutu [ELEMEN] Kebijakan, sta...",40,,,,[STANDAR] Standar Pengelolaan Pembelajaran [PE...,0.680127,Mirip
6,3,1,Budaya Mutu,Efektifitas pelaksanaan kegiatan terkait siste...,Efektifitas pelaksanaan \nkegiatan terkait \ns...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.764388,Mirip
7,3,2,Budaya Mutu,Efektifitas pelaksanaan kegiatan terkait siste...,Efektifitas pelaksanaan \nkegiatan terkait \ns...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,303,,,,[STANDAR] Standar Tata Kelola dan Tata Pamong ...,0.730314,Mirip
8,3,3,Budaya Mutu,Efektifitas pelaksanaan kegiatan terkait siste...,Efektifitas pelaksanaan \nkegiatan terkait \ns...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,304,,,,[STANDAR] Standar Tata Kelola dan Tata Pamong ...,0.730314,Mirip
9,4,1,Budaya Mutu,Efektifitas pelaksanaan standar dan indikator ...,Efektifitas pelaksanaan kegiatan terkait stand...,[KRITERIA] Budaya Mutu [ELEMEN] Efektifitas pe...,223,,,,[STANDAR] Standar Implementasi Mbkm [PERNYATAA...,0.724490,Mirip


In [35]:
hasil_top3_df.to_excel("hasil_top3_word2vec_cosine_similarity.xlsx", index=False)

files.download("hasil_top3_word2vec_cosine_similarity.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>